### Aggregation and Grouping
### A fundamental piece of many data analysis tasks is efficient summarization: computing aggregations like sum, mean, median, min, and max, in which a single number summarizes aspects of a potentially large dataset. In this chapter, we'll explore aggregations in Pandas, from simple operations like what we've seen on NumPy arrays to more sophisticated operations based on the concept of a groupby.

### For convenience, we'll use the same display magic function that we used in the previous chapters:

In [2]:
import numpy as np
import pandas as pd

class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args

    def _repr_html_(self):
        return '\n'.join(self.template.format(a, eval(a)._repr_html_())
                         for a in self.args)

    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                           for a in self.args)

### Planets Data

In [3]:
import seaborn as sns
planets = sns.load_dataset('planets')
planets.shape

(1035, 6)

In [3]:
%pip install seaborn

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 55.4 MB/s eta 0:00:01
   --- ------------------------------------ 0.8/9.5 MB 1.4 MB/s eta 0:00:07
   ---- ----------------------------------- 1.0/9.5 MB 1.7 MB/s eta 0:00:06
   ------ --------------------------------- 1.6/9.5 MB 1.5 MB/s eta 0:00:06
   ------ --------------------------------- 1.6/9.5 MB 1.5 MB/s eta 0:00:06
   -------- ------------------------------- 2.1/9.5 MB 1.4 MB/s eta 0:00:06
   ----------- ---------------------------- 2.6/9.5 MB 1.6 MB/s eta 0:00:05
   -------------- ------------------------- 3.4/9.5 MB 1.7 MB/s eta 0:00:04
   ----------------- ---------------------- 4.2/9.5 MB 1.9 MB/s eta 0:00:03
   ------------------- -------------------- 4.7/9.5 MB 2.0 MB/s eta 0:00:03
   ----------------------- ---------------- 5.5/9.5 MB 2.1 MB/s eta 0:00:02
   ------------------------- -------------- 6.0/9.5 MB 2.1 MB/s eta 0:00:02
   ---------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
planets.head()

,method,number,orbital_period,mass,distance,year
0,Radial Velocity,1,269.300,7.10,77.40,2006
1,Radial Velocity,1,874.774,2.21,56.95,2008
2,Radial Velocity,1,763.000,2.60,19.84,2011
3,Radial Velocity,1,326.030,19.40,110.62,2007
4,Radial Velocity,1,516.220,10.50,119.47,2009


In [3]:
planets.info()

<class 'pandas.DataFrame'>
RangeIndex: 1035 entries, 0 to 1034
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   method          1035 non-null   str    
 1   number          1035 non-null   int64  
 2   orbital_period  992 non-null    float64
 3   mass            513 non-null    float64
 4   distance        808 non-null    float64
 5   year            1035 non-null   int64  
dtypes: float64(3), int64(2), str(1)
memory usage: 60.5 KB


In [4]:
planets['method'].unique()

<ArrowStringArray>
[              'Radial Velocity',                       'Imaging',
     'Eclipse Timing Variations',                       'Transit',
                    'Astrometry',     'Transit Timing Variations',
 'Orbital Brightness Modulation',                  'Microlensing',
                 'Pulsar Timing',   'Pulsation Timing Variations']
Length: 10, dtype: str

### Simple Aggregation in Pandas
##### In "Aggregations: Min, Max, and Everything In Between", we explored some of the data aggregations available for NumPy arrays. As with a one-dimensional NumPy array, for a Pandas Series the aggregates return a single value:

In [4]:
rng = np.random.RandomState(42)
ser = pd.Series(rng.rand(5))
ser

0    0.374540
1    0.950714
2    0.731994
3    0.598658
4    0.156019
dtype: float64

In [ ]:
#find the sum of the series 
ser.sum()

np.float64(2.811925491708157)

In [6]:
#find the mean of the series 
ser.mean()

np.float64(0.5623850983416314)

In [7]:
# return results in each column 
df = pd.DataFrame({'A': rng.rand(5),
                   'B': rng.rand(5)})
df

,A,B
0,0.155995,0.020584
1,0.058084,0.969910
2,0.866176,0.832443
3,0.601115,0.212339
4,0.708073,0.181825


In [8]:
df.mean()

A    0.477888
B    0.443420
dtype: float64

In [9]:
# get the mean by specifying the axis
df.mean(axis=1)

0    0.088290
1    0.513997
2    0.849309
3    0.406727
4    0.444949
dtype: float64

In [10]:
planets.info()

<class 'pandas.DataFrame'>
RangeIndex: 1035 entries, 0 to 1034
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   method          1035 non-null   str    
 1   number          1035 non-null   int64  
 2   orbital_period  992 non-null    float64
 3   mass            513 non-null    float64
 4   distance        808 non-null    float64
 5   year            1035 non-null   int64  
dtypes: float64(3), int64(2), str(1)
memory usage: 60.5 KB


In [11]:
planets['year'].unique()

array([2006, 2008, 2011, 2007, 2009, 2002, 1996, 2010, 2001, 1995, 2004,
       2012, 2013, 2005, 2000, 2003, 1997, 1999, 2014, 1998, 1989, 1992,
       1994])

In [12]:
planets.isnull().sum()

method              0
number              0
orbital_period     43
mass              522
distance          227
year                0
dtype: int64

In [13]:
planets.dropna().describe()

,number,orbital_period,mass,distance,year
count,498.00000,498.000000,498.000000,498.000000,498.000000
mean,1.73494,835.778671,2.509320,52.068213,2007.377510
std,1.17572,1469.128259,3.636274,46.596041,4.167284
min,1.00000,1.328300,0.003600,1.350000,1989.000000
25%,1.00000,38.272250,0.212500,24.497500,2005.000000
50%,1.00000,357.000000,1.245000,39.940000,2009.000000
75%,2.00000,999.600000,2.867500,59.332500,2011.000000
max,6.00000,17337.500000,25.000000,354.000000,2014.000000


### groupby: Split, Apply, Combine
#### Simple aggregations can give you a flavor of your dataset, but often we would prefer to aggregate conditionally on some label or index: this is implemented in the so-called groupby operation. The name "group by" comes from a command in the SQL database language, but it is perhaps more illuminative to think of it in the terms first coined by Hadley Wickham of Rstats fame: split, apply, combine.

### Split, Apply, Combine
#### A canonical example of this split-apply-combine operation, where the "apply" is a summation aggregation, is illustrated in this figure:
#### This illustrates what the groupby operation accomplishes:

#### The split step involves breaking up and grouping a DataFrame depending on the value of the specified key.
#### The apply step involves computing some function, usually an aggregate, transformation, or filtering, within the individual groups.
#### The combine step merges the results of these operations into an output array.
#### While this could certainly be done manually using some combination of the masking, aggregation, and merging commands covered earlier, an important realization is that the intermediate splits do not need to be explicitly instantiated. Rather, the groupby can (often) do this in a single pass over the data, updating the sum, mean, count, min, or other aggregate for each group along the way. The power of the groupby is that it abstracts away these steps: the user need not think about how the computation is done under the hood, but rather can think about the operation as a whole.

#### As a concrete example, let's take a look at using Pandas for the computation shown in the following figure. We'll start by creating the input DataFrame:

In [14]:
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data': range(6)}, columns=['key', 'data'])
df

,key,data
0,A,0
1,B,1
2,C,2
3,A,3
4,B,4
5,C,5


In [15]:
# use the groupby keyword
df.groupby('key')

In [16]:
df.groupby('key').sum()

,data
key,
A,3
B,5
C,7


### Column indexing
#### The GroupBy object supports column indexing in the same way as the DataFrame, and returns a modified GroupBy object. For example:

In [17]:
planets.head()

,method,number,orbital_period,mass,distance,year
0,Radial Velocity,1,269.300,7.10,77.40,2006
1,Radial Velocity,1,874.774,2.21,56.95,2008
2,Radial Velocity,1,763.000,2.60,19.84,2011
3,Radial Velocity,1,326.030,19.40,110.62,2007
4,Radial Velocity,1,516.220,10.50,119.47,2009


In [18]:
planets.method.unique()

<ArrowStringArray>
[              'Radial Velocity',                       'Imaging',
     'Eclipse Timing Variations',                       'Transit',
                    'Astrometry',     'Transit Timing Variations',
 'Orbital Brightness Modulation',                  'Microlensing',
                 'Pulsar Timing',   'Pulsation Timing Variations']
Length: 10, dtype: str

In [19]:
planets.groupby('method')

In [20]:
planets.groupby('method')['orbital_period'].median()

method
Astrometry                         631.180000
Eclipse Timing Variations         4343.500000
Imaging                          27500.000000
Microlensing                      3300.000000
Orbital Brightness Modulation        0.342887
Pulsar Timing                       66.541900
Pulsation Timing Variations       1170.000000
Radial Velocity                    360.200000
Transit                              5.714932
Transit Timing Variations           57.011000
Name: orbital_period, dtype: float64

In [21]:
planets.groupby('method')[['orbital_period', 'year']].median()

,orbital_period,year
method,,
Astrometry,631.180000,2011.5
Eclipse Timing Variations,4343.500000,2010.0
Imaging,27500.000000,2009.0
Microlensing,3300.000000,2010.0
Orbital Brightness Modulation,0.342887,2011.0
Pulsar Timing,66.541900,1994.0
Pulsation Timing Variations,1170.000000,2007.0
Radial Velocity,360.200000,2009.0
Transit,5.714932,2012.0


### Iteration over groups
#### The GroupBy object supports direct iteration over the groups, returning each group as a Series or DataFrame:

In [22]:
planets.groupby('method').size()

method
Astrometry                         2
Eclipse Timing Variations          9
Imaging                           38
Microlensing                      23
Orbital Brightness Modulation      3
Pulsar Timing                      5
Pulsation Timing Variations        1
Radial Velocity                  553
Transit                          397
Transit Timing Variations          4
dtype: int64

### Using the groupby method using dispatch methods 

In [23]:
planets.groupby('method')['year'].describe().unstack()

       method                       
count  Astrometry                          2.0
       Eclipse Timing Variations           9.0
       Imaging                            38.0
       Microlensing                       23.0
       Orbital Brightness Modulation       3.0
                                         ...  
max    Pulsar Timing                    2011.0
       Pulsation Timing Variations      2007.0
       Radial Velocity                  2014.0
       Transit                          2014.0
       Transit Timing Variations        2014.0
Length: 80, dtype: float64

### Aggregate, Filter, Transform, Apply

In [24]:
rng = np.random.RandomState(0)
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                    'data1': range(6),
                    'data2': rng.randint(0, 10, 6)}, 
                    columns=['key', 'data1', 'data2'])
df

,key,data1,data2
0,A,0,5
1,B,1,0
2,C,2,3
3,A,3,3
4,B,4,7
5,C,5,9


### Aggregation
#### Aggregation allows more flexibility than groupby 

In [25]:
df.groupby('key').aggregate(['min', np.median, max])

data1            data2           
      min median max   min median max
key                                  
A       0    1.5   3     3    4.0   5
B       1    2.5   4     0    3.5   7
C       2    3.5   5     3    6.0   9

In [26]:
# mapping column names 
df

,key,data1,data2
0,A,0,5
1,B,1,0
2,C,2,3
3,A,3,3
4,B,4,7
5,C,5,9


In [27]:
df.groupby('key').aggregate({'data1': 'min',
                             'data2': 'max'})

,data1,data2
key,,
A,0,5
B,1,7
C,2,9


### Filtering
#### A filtering operation allows you to drop data based on the group properties.

In [28]:
df.head()

,key,data1,data2
0,A,0,5
1,B,1,0
2,C,2,3
3,A,3,3
4,B,4,7


In [29]:
def filter_func(x):
    return x['data2'].std() > 4

display('df', "df.groupby('key').std()", 
        "df.groupby('key').filter(filter_func)")

df
  key  data1  data2
0   A      0      5
1   B      1      0
2   C      2      3
3   A      3      3
4   B      4      7
5   C      5      9

df.groupby('key').std()
       data1     data2
key                   
A    2.12132  1.414214
B    2.12132  4.949747
C    2.12132  4.242641

df.groupby('key').filter(filter_func)
  key  data1  data2
1   B      1      0
2   C      2      3
4   B      4      7
5   C      5      9

###  Transformation
#### Transformation can return some transformed version of the full data to recombine.

In [30]:
df

,key,data1,data2
0,A,0,5
1,B,1,0
2,C,2,3
3,A,3,3
4,B,4,7
5,C,5,9


In [31]:
def center(x):
    return x - x.mean()
df.groupby('key').transform(center)

,data1,data2
0,-1.5,1.0
1,-1.5,-3.5
2,-1.5,-3.0
3,1.5,-1.0
4,1.5,3.5
5,1.5,3.0


### The apply method
#### The apply method lets you apply an arbitrary function to the group results

In [32]:
df

,key,data1,data2
0,A,0,5
1,B,1,0
2,C,2,3
3,A,3,3
4,B,4,7
5,C,5,9


In [33]:
def norm_by_data2(x):
    # x is a DataFrame of group values
    x['data1'] /= x['data2'].sum()
    return x

df.groupby('key').apply(norm_by_data2)

data1  data2
key                   
A   0  0.000000      5
    3  0.375000      3
B   1  0.142857      0
    4  0.571429      7
C   2  0.166667      3
    5  0.416667      9

### Specifying the Split Key
#### A list, array, series, or index providing the grouping keys
#### The key can be any series or list with a length matching that of the DataFram

In [34]:
df

,key,data1,data2
0,A,0,5
1,B,1,0
2,C,2,3
3,A,3,3
4,B,4,7
5,C,5,9


In [35]:
L = [0, 1, 0, 1, 2, 0]
print(L)
L_grouped = df.groupby(L).sum()
print(L_grouped)

[0, 1, 0, 1, 2, 0]
   key  data1  data2
0  ACC      7     17
1   BA      4      3
2    B      4      7


In [36]:
df.groupby(df['key']).sum()

,data1,data2
key,,
A,3,8
B,5,7
C,7,12


### A dictionary or series mapping index to group

In [37]:
df2 = df.set_index('key')
mapping = {'A': 'vowel', 'B': 'consonant', 'C': 'consonant'}
display('df2', 'df2.groupby(mapping).sum()')

,data1,data2
key,,
A,0,5
B,1,0
C,2,3
A,3,3
B,4,7
C,5,9
,data1,data2
key,,
consonant,12,19


#### Any Python Functiom

In [38]:
df2

,data1,data2
key,,
A,0,5
B,1,0
C,2,3
A,3,3
B,4,7
C,5,9


In [ ]:
df2.groupby(str.lower).mean()

### A list of valid keys
Further, any of the preceding key choices can be combined to group on a multi-index:

In [ ]:
df2.groupby([str.lower, mapping]).mean()

In [ ]:
decade = 10 * (planets['year'] // 10)
decade = decade.astype(str) + 's'
decade.name = 'decade'
planets.groupby(['method', decade])['number'].sum().unstack().fillna(0)